# Figure 4: forward-genetic query

KRAS suppression in colon cancer, the RAS GenePT neighbourhood, and the forward-genetic experimental-rank panel.

In [ ]:
from pxfquery import PxFQuery

question = "In colon cancer, what functional programs change after genetic suppression of KRAS?"
pxf = PxFQuery()
qdata = pxf.tl.parse(question)
pxf.tl.answer(qdata)
answer = pxf.get.answer(qdata)
print(answer)


## RAS GenePT neighbourhood

In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "DejaVu Sans"

PROJECT_ROOT = Path("/Users/dudu/Documents/3_Project/12_PxFquery")
EMBEDDING_PATH = PROJECT_ROOT / "2_project_asset/1_raw_material/legacy_flat_asset_library_v20260614/data/genept_embeddings/M-0197_gene_embedding_m3_filtered.npz"
GENE_NAMES_PATH = PROJECT_ROOT / "2_project_asset/1_raw_material/legacy_flat_asset_library_v20260614/data/genept_embeddings/M-0198_gene_names_m3_filtered.csv"
NEIGHBORS_PATH = Path("/Users/dudu/.cache/pxfquery/resources/v20260628/gene_neighbors.json")


def load_genept() -> tuple[list[str], np.ndarray]:
    names = pd.read_csv(GENE_NAMES_PATH, header=None)[0].astype(str).tolist()
    emb = np.load(EMBEDDING_PATH)["data"]
    if len(names) != emb.shape[0]:
        raise ValueError(f"gene name count {len(names)} does not match embedding rows {emb.shape[0]}")
    return names, emb


def selected_genes(neighbors: dict[str, list], anchors: list[str], top_n: int) -> list[str]:
    genes: list[str] = []
    for anchor in anchors:
        if anchor not in genes:
            genes.append(anchor)
        for name, _score in neighbors.get(anchor, [])[:top_n]:
            if name not in genes:
                genes.append(name)
    return genes


def pca2(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    x = x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)
    x = x - x.mean(axis=0, keepdims=True)
    _u, _s, vt = np.linalg.svd(x, full_matrices=False)
    coords = x @ vt[:2].T
    coords = coords / np.maximum(np.abs(coords).max(axis=0, keepdims=True), 1e-8)
    return coords


def cosine_edges(genes: list[str], neighbors: dict[str, list], min_cosine: float, anchors: set[str]) -> list[tuple[str, str, float]]:
    gene_set = set(genes)
    seen = set()
    edges = []
    for gene in genes:
        if gene not in anchors:
            continue
        for nbr, score_int in neighbors.get(gene, []):
            score = float(score_int) / 100.0
            if score < min_cosine:
                break
            if nbr not in gene_set:
                continue
            key = tuple(sorted((gene, nbr)))
            if key in seen or gene == nbr:
                continue
            seen.add(key)
            edges.append((gene, nbr, score))
    return edges


def render(output_dir: Path, top_n: int = 18, min_edge_cosine: float = 0.66) -> tuple[Path, Path]:
    anchors = ["KRAS", "NRAS", "HRAS"]
    neighbors = json.loads(NEIGHBORS_PATH.read_text())
    genes = selected_genes(neighbors, anchors, top_n=top_n)
    names, emb = load_genept()
    index = {name: i for i, name in enumerate(names)}
    genes = [gene for gene in genes if gene in index]
    x = np.asarray([emb[index[gene]] for gene in genes])
    coords = pca2(x)
    pos = {gene: coords[i] for i, gene in enumerate(genes)}
    edges = cosine_edges(genes, neighbors, min_cosine=min_edge_cosine, anchors=set(anchors))
    # KRAS and NRAS are near-identical in the local projection. Separate them
    # slightly for label readability while preserving their visual proximity.
    if "KRAS" in pos and "NRAS" in pos:
        delta = np.asarray([0.045, -0.02])
        pos["KRAS"] = pos["KRAS"] - delta
        pos["NRAS"] = pos["NRAS"] + delta

    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    ax.set_axis_off()

    for a, b, score in edges:
        xa, ya = pos[a]
        xb, yb = pos[b]
        lw = 0.45 + 2.2 * (score - min_edge_cosine) / max(1e-6, 1.0 - min_edge_cosine)
        color = "#d97706" if a in {"KRAS", "NRAS", "HRAS"} and b in {"KRAS", "NRAS", "HRAS"} else "#9ca3af"
        alpha = 0.58 if color == "#d97706" else 0.24
        ax.plot([xa, xb], [ya, yb], color=color, lw=lw, alpha=alpha, zorder=1)

    route_genes = {"KRAS", "NRAS", "HRAS"}
    ras_related = {gene for gene in genes if gene.startswith(("RAS", "RAP", "RAL", "RRAS")) or gene in {"SOS1", "SOS2", "NF1"}}
    for gene in genes:
        x0, y0 = pos[gene]
        if gene in route_genes:
            ax.scatter([x0], [y0], s=420, facecolor="#fef3c7", edgecolor="#d97706", linewidth=2.0, zorder=4)
        elif gene in ras_related:
            ax.scatter([x0], [y0], s=120, facecolor="#e0f2fe", edgecolor="#0284c7", linewidth=1.0, zorder=3)
        else:
            ax.scatter([x0], [y0], s=58, facecolor="#f3f4f6", edgecolor="#9ca3af", linewidth=0.8, zorder=2)

    label_genes = route_genes | {"SOS1", "SOS2", "NF1", "RRAS2", "RAP1A", "RASAL2", "RASGRP1", "RALGDS"}
    label_offsets = {
        "KRAS": (-0.055, -0.065),
        "NRAS": (0.060, 0.060),
        "HRAS": (0.000, 0.070),
        "RAP1A": (0.020, 0.050),
        "RASAL2": (0.000, 0.065),
        "RASGRP1": (0.000, 0.060),
    }
    for gene in genes:
        if gene not in label_genes:
            continue
        x0, y0 = pos[gene]
        dx, dy = label_offsets.get(gene, (0.0, 0.045))
        ax.text(x0 + dx, y0 + dy, gene, ha="center", va="bottom", fontsize=9 if gene in route_genes else 7.2, weight="bold" if gene in route_genes else "normal", zorder=5)

    ax.set_title("GenePT semantic neighborhood of KRAS-family routes", fontsize=11.5, weight="bold", pad=10)
    ax.text(
        0.02,
        0.02,
        "GenePT model-3 local projection; edges show anchor semantic-neighbor cosine similarities",
        transform=ax.transAxes,
        fontsize=7,
        color="#4b5563",
        ha="left",
        va="bottom",
    )
    ax.margins(0.18)

    out_pdf = output_dir / "fig4c_genept_kras_family_semantic_map.pdf"
    out_png = output_dir / "fig4c_genept_kras_family_semantic_map.png"
    fig.savefig(out_pdf, bbox_inches="tight")
    fig.savefig(out_png, dpi=240, bbox_inches="tight")
    plt.close(fig)
    return out_pdf, out_png


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-dir", type=Path, default=Path("4_artifact/4_picture/figure4/panel_C_genept_semantic_map"))
    parser.add_argument("--top-n", type=int, default=14)
    parser.add_argument("--min-edge-cosine", type=float, default=0.68)
    args = parser.parse_args()
    for path in render(args.output_dir, top_n=args.top_n, min_edge_cosine=args.min_edge_cosine):
        print(path)


if __name__ == "__main__":
    main()


## Forward-genetic benchmark panel

In [ ]:
from __future__ import annotations

import argparse
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon


PXF_COLOR = "#d62728"
LLM_COLOR = "#1f77b4"
SYSTEMS = ["PxFquery", "Direct LLM"]

FORWARD_PANELS = [
    ("activated_top3", "Activated top3"),
    ("suppressed_top3", "Suppressed top3"),
    ("activated_top5", "Activated top5"),
    ("suppressed_top5", "Suppressed top5"),
]
REVERSE_PANELS = [
    ("recommendations_top1", "Top1"),
    ("recommendations_top3", "Top3"),
    ("recommendations_top5", "Top5"),
    ("recommendations_top10", "Top10"),
]


@dataclass(frozen=True)
class DatasetSpec:
    figure_id: str
    task_type: str
    source_name: str
    output_stem: str
    panels: list[tuple[str, str]]


DATASETS = [
    DatasetSpec(
        figure_id="fig4_panel_b",
        task_type="forward_genetic",
        source_name="forward_genetic_xpr100_first100",
        output_stem="fig4_panel_b_forward_genetic_xpr",
        panels=FORWARD_PANELS,
    ),
    DatasetSpec(
        figure_id="fig5_panel_b",
        task_type="forward_drug",
        source_name="forward_drug_100_first100",
        output_stem="fig5_panel_b_forward_drug",
        panels=FORWARD_PANELS,
    ),
    DatasetSpec(
        figure_id="fig6_panel_b",
        task_type="reverse_genetic",
        source_name="reverse_genetic_100",
        output_stem="fig6_panel_b_reverse_genetic",
        panels=REVERSE_PANELS,
    ),
    DatasetSpec(
        figure_id="fig7_panel_b",
        task_type="reverse_drug",
        source_name="reverse_drug_100",
        output_stem="fig7_panel_b_reverse_drug",
        panels=REVERSE_PANELS,
    ),
]


def configure_matplotlib() -> None:
    mpl.rcParams.update(
        {
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "font.family": "Arial",
            "axes.unicode_minus": False,
        }
    )


def query_order(query_id: str) -> int:
    match = re.search(r"_(\d+)$", str(query_id))
    return int(match.group(1)) if match else 10**9


def pvalue_stars(pvalue: float | None) -> str:
    if pvalue is None or not np.isfinite(pvalue):
        return "n.s."
    if pvalue < 1e-4:
        return "****"
    if pvalue < 1e-3:
        return "***"
    if pvalue < 1e-2:
        return "**"
    if pvalue < 0.05:
        return "*"
    return "n.s."


def paired_pvalue(data: pd.DataFrame) -> tuple[float | None, int]:
    pivot = data.pivot_table(
        index="query_id",
        columns="system",
        values="median_rank_percentile",
        aggfunc="first",
    ).dropna(subset=SYSTEMS)
    if len(pivot) < 2:
        return None, len(pivot)
    diff = pivot["PxFquery"].astype(float) - pivot["Direct LLM"].astype(float)
    if np.allclose(diff.to_numpy(), 0):
        return 1.0, len(pivot)
    return float(wilcoxon(pivot["PxFquery"], pivot["Direct LLM"]).pvalue), len(pivot)


def first100_canonical_ids(answers_path: Path) -> list[str]:
    ids: list[str] = []
    with answers_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            gate = record.get("evidence_resolution_gate") or {}
            rows = ((record.get("answer_tables") or {}).get("ranked_results") or [])
            if gate.get("answer_unresolved") is False and rows:
                ids.append(str(record.get("query_id")))
    return sorted(ids, key=query_order)[:100]


def build_forward_query_median(checks_path: Path, query_ids: Iterable[str]) -> pd.DataFrame:
    selected_ids = {str(query_id) for query_id in query_ids}
    checks = pd.read_csv(checks_path)
    checks = checks[checks["query_id"].astype(str).isin(selected_ids)].copy()
    checks = checks[checks["not_found"].astype(str).str.lower() != "true"].copy()
    checks["rank_percentile"] = pd.to_numeric(checks["rank_percentile"], errors="coerce")
    return (
        checks.dropna(subset=["rank_percentile"])
        .groupby(["query_id", "task_type", "field", "system"], dropna=False)
        .agg(
            median_rank_percentile=("rank_percentile", "median"),
            mean_rank_percentile=("rank_percentile", "mean"),
            n_found=("rank_percentile", "size"),
            same_direction_count=("same_direction", lambda values: int(values.astype(str).str.lower().eq("true").sum())),
        )
        .reset_index()
    )


def write_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def prepare_release_data(task_root: Path, release_root: Path) -> None:
    table_root = task_root / "4_artifact/5_table"
    data_dir = release_root / "data"

    forward_sources = [
        (
            "forward_genetic_xpr100_first100",
            "fg_xpr100_resolved_official_t144_20260703",
        ),
        (
            "forward_drug_100_first100",
            "fd100_resolved_official_t144_20260703",
        ),
    ]
    for output_name, source_dir_name in forward_sources:
        source_dir = table_root / source_dir_name
        query_ids = first100_canonical_ids(source_dir / "filtered_batch_pxfquery_answers.jsonl")
        query_median = build_forward_query_median(source_dir / "filtered_batch_checks.csv", query_ids)
        write_csv(pd.DataFrame({"query_id": query_ids}), data_dir / "query_ids" / f"{output_name}_query_ids.csv")
        write_csv(query_median, data_dir / "query_medians" / f"{output_name}_query_median.csv")

    reverse = pd.read_csv(table_root / "rg_rd_batch100_current/rg_rd_batch100_current_query_median.csv")
    for task_type, output_name in [
        ("reverse_genetic", "reverse_genetic_100"),
        ("reverse_drug", "reverse_drug_100"),
    ]:
        query_median = reverse[reverse["task_type"] == task_type].copy()
        query_ids = sorted(query_median["query_id"].astype(str).unique(), key=query_order)
        write_csv(pd.DataFrame({"query_id": query_ids}), data_dir / "query_ids" / f"{output_name}_query_ids.csv")
        write_csv(query_median, data_dir / "query_medians" / f"{output_name}_query_median.csv")

    source_manifest = pd.DataFrame(
        [
            {
                "dataset": "forward_genetic_xpr100_first100",
                "source": str(table_root / "fg_xpr100_resolved_official_t144_20260703"),
                "selection": "first 100 canonical evaluable xpr forward genetic queries with nonempty ranked evidence table",
            },
            {
                "dataset": "forward_drug_100_first100",
                "source": str(table_root / "fd100_resolved_official_t144_20260703"),
                "selection": "first 100 canonical evaluable forward drug queries with nonempty ranked evidence table",
            },
            {
                "dataset": "reverse_genetic_100",
                "source": str(table_root / "rg_rd_batch100_current/rg_rd_batch100_current_query_median.csv"),
                "selection": "existing reverse genetic benchmark query-median table",
            },
            {
                "dataset": "reverse_drug_100",
                "source": str(table_root / "rg_rd_batch100_current/rg_rd_batch100_current_query_median.csv"),
                "selection": "existing reverse drug benchmark query-median table",
            },
        ]
    )
    write_csv(source_manifest, data_dir / "audit" / "source_manifest.csv")


def draw_panel(ax: plt.Axes, data: pd.DataFrame, title: str) -> dict[str, object]:
    values_by_system = [
        data[data["system"] == system]["median_rank_percentile"].dropna().astype(float).to_numpy()
        for system in SYSTEMS
    ]
    violin = ax.violinplot(
        values_by_system,
        positions=[0, 1],
        widths=0.72,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for body, color in zip(violin["bodies"], [PXF_COLOR, LLM_COLOR]):
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.16)
        body.set_linewidth(0.9)

    for x, vals in enumerate(values_by_system):
        if len(vals):
            q1, median, q3 = np.percentile(vals, [25, 50, 75])
            ax.vlines(x, q1, q3, color="#555555", linewidth=2.2, alpha=0.58, zorder=3)
            ax.hlines(q1, x - 0.13, x + 0.13, color="#555555", linewidth=1.1, alpha=0.58, zorder=3)
            ax.hlines(q3, x - 0.13, x + 0.13, color="#555555", linewidth=1.1, alpha=0.58, zorder=3)
            ax.hlines(median, x - 0.18, x + 0.18, color="black", linewidth=1.25, zorder=4)

    rng = np.random.default_rng(20260702)
    for x, color, vals in zip([0, 1], [PXF_COLOR, LLM_COLOR], values_by_system):
        jitter = rng.uniform(-0.13, 0.13, len(vals))
        ax.scatter(
            np.full(len(vals), x) + jitter,
            vals,
            s=16,
            c=color,
            edgecolor="white",
            linewidth=0.22,
            alpha=0.36,
            zorder=5,
        )

    pvalue, paired_n = paired_pvalue(data)
    significance = pvalue_stars(pvalue)
    ax.text(0.5, 0.985, significance, transform=ax.transAxes, ha="center", va="top", fontsize=10)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(SYSTEMS, fontsize=8)
    ax.set_ylim(105, -5)
    ax.grid(axis="y", alpha=0.22)
    return {
        "paired_n": paired_n,
        "paired_wilcoxon_p": pvalue,
        "significance": significance,
        "pxf_median": float(np.nanmedian(values_by_system[0])) if len(values_by_system[0]) else np.nan,
        "llm_median": float(np.nanmedian(values_by_system[1])) if len(values_by_system[1]) else np.nan,
    }


def plot_dataset(release_root: Path, spec: DatasetSpec) -> pd.DataFrame:
    query_median_path = release_root / "data/query_medians" / f"{spec.source_name}_query_median.csv"
    data = pd.read_csv(query_median_path)
    data = data[data["task_type"] == spec.task_type].copy()
    data["median_rank_percentile"] = pd.to_numeric(data["median_rank_percentile"], errors="coerce")

    fig, axes = plt.subplots(2, 2, figsize=(6.6, 5.8), sharey=True)
    summary_rows = []
    for ax, (field, title) in zip(axes.ravel(), spec.panels):
        panel_data = data[data["field"] == field].copy()
        row = draw_panel(ax, panel_data, title)
        row.update(
            {
                "figure_id": spec.figure_id,
                "task_type": spec.task_type,
                "field": field,
                "field_title": title,
            }
        )
        summary_rows.append(row)
    for ax in axes.ravel()[::2]:
        ax.set_ylabel("Experimental rank percentile (%)")
    fig.tight_layout()

    pdf_path = release_root / "figures/pdf" / f"{spec.output_stem}.pdf"
    png_path = release_root / "figures/png" / f"{spec.output_stem}.png"
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    png_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return pd.DataFrame(summary_rows)


def plot_release_figures(release_root: Path) -> pd.DataFrame:
    configure_matplotlib()
    summaries = [plot_dataset(release_root, spec) for spec in DATASETS]
    summary = pd.concat(summaries, ignore_index=True)
    write_csv(summary, release_root / "data/audit/panel_b_summary.csv")

    figure_manifest = pd.DataFrame(
        [
            {
                "figure_id": spec.figure_id,
                "task_type": spec.task_type,
                "source_name": spec.source_name,
                "pdf": str(Path("figures/pdf") / f"{spec.output_stem}.pdf"),
                "png": str(Path("figures/png") / f"{spec.output_stem}.png"),
            }
            for spec in DATASETS
        ]
    )
    write_csv(figure_manifest, release_root / "data/audit/figure_manifest.csv")
    return summary


def parse_args() -> argparse.Namespace:
    default_release_root = Path(__file__).resolve().parents[1]
    default_task_root = default_release_root.parents[2]
    parser = argparse.ArgumentParser(description="Prepare and plot PxFquery panel B benchmark figures.")
    parser.add_argument("--task-root", type=Path, default=default_task_root)
    parser.add_argument("--release-root", type=Path, default=default_release_root)
    parser.add_argument("--mode", choices=["all", "prepare", "plot"], default="all")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    if args.mode in {"all", "prepare"}:
        prepare_release_data(args.task_root, args.release_root)
    if args.mode in {"all", "plot"}:
        summary = plot_release_figures(args.release_root)
        print(summary.to_string(index=False))


if __name__ == "__main__":
    main()


## Evidence network graph\n\nThe original rendering code retained for this figure.

In [ ]:
from __future__ import annotations

import argparse
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def _short_label(text: str, max_len: int = 26) -> str:
    text = str(text).replace("HALLMARK_", "").replace("_", " ")
    text = text.replace("  ", " ").strip()
    if len(text) <= max_len:
        return text
    return text[: max_len - 1] + "..."


def _spread_positions(items: list[str], x: float, y_top: float = 0.92, y_bottom: float = 0.12) -> dict[str, tuple[float, float]]:
    if not items:
        return {}
    if len(items) == 1:
        return {items[0]: (x, (y_top + y_bottom) / 2)}
    step = (y_top - y_bottom) / (len(items) - 1)
    return {item: (x, y_top - i * step) for i, item in enumerate(items)}


def render_case(case_dir: Path, case_id: str, output_dir: Path, *, max_functions: int = 10, max_edges_per_route: int = 3) -> Path:
    route_path = case_dir / f"{case_id}_route_summary.csv"
    rf_path = case_dir / f"{case_id}_route_function_results.csv"
    if not route_path.exists() or not rf_path.exists():
        raise FileNotFoundError(f"missing route files for {case_id}")

    routes = pd.read_csv(route_path)
    rf = pd.read_csv(rf_path)
    routes = routes[routes["status"].eq("executed")].copy()
    if routes.empty:
        routes = pd.read_csv(route_path)

    # Pick consensus-readable functions, then only draw per-route edges to those terms.
    ranked_path = case_dir / f"{case_id}_ranked_results.csv"
    if ranked_path.exists():
        ranked = pd.read_csv(ranked_path).head(max_functions)
        selected_functions = list(ranked["label"].astype(str))
    else:
        selected_functions = list(
            rf.assign(abs_score=rf["score"].abs())
            .sort_values("abs_score", ascending=False)["label"]
            .drop_duplicates()
            .head(max_functions)
        )
    selected = rf[rf["label"].astype(str).isin(selected_functions)].copy()

    cells = list(dict.fromkeys(routes["cell"].dropna().astype(str)))
    perts = list(dict.fromkeys(routes["perturbation"].dropna().astype(str)))
    funcs = selected_functions

    pos = {}
    pos.update({f"cell:{k}": v for k, v in _spread_positions(cells, 0.12).items()})
    pos.update({f"pert:{k}": v for k, v in _spread_positions(perts, 0.48).items()})
    pos.update({f"func:{k}": v for k, v in _spread_positions(funcs, 0.86, y_top=0.96, y_bottom=0.08).items()})

    fig_h = max(5.0, min(12.0, 0.45 * max(len(cells), len(perts), len(funcs)) + 2.8))
    fig, ax = plt.subplots(figsize=(11, fig_h))
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.text(0.12, 1.02, "Cell context", ha="center", va="bottom", fontsize=12, weight="bold")
    ax.text(0.48, 1.02, "Perturbation evidence", ha="center", va="bottom", fontsize=12, weight="bold")
    ax.text(0.86, 1.02, "Functional programs", ha="center", va="bottom", fontsize=12, weight="bold")

    # Cell -> perturbation route edges: neutral; direct/semantic route type is shown by line style.
    for _, row in routes.iterrows():
        c = f"cell:{row['cell']}"
        p = f"pert:{row['perturbation']}"
        if c not in pos or p not in pos:
            continue
        style = "-" if str(row.get("perturbation_match_type", "")).startswith(("user", "normalized")) else "--"
        ax.annotate(
            "",
            xy=pos[p],
            xytext=pos[c],
            arrowprops=dict(arrowstyle="-", color="#9aa3ad", lw=1.2, linestyle=style, alpha=0.62),
        )

    # Perturbation -> function edges: direction is encoded on the edge.
    edge_rows = []
    for route_id, block in selected.groupby("route_id", sort=False):
        block = block.assign(abs_score=block["score"].abs()).sort_values("abs_score", ascending=False).head(max_edges_per_route)
        edge_rows.append(block)
    draw_edges = pd.concat(edge_rows, ignore_index=True) if edge_rows else selected.head(0)
    max_abs = max(1.0, float(draw_edges["score"].abs().max() if not draw_edges.empty else 1.0))
    for _, row in draw_edges.iterrows():
        p = f"pert:{row['perturbation']}"
        f = f"func:{row['label']}"
        if p not in pos or f not in pos:
            continue
        score = float(row["score"])
        color = "#c94747" if score > 0 else "#2f6fa3"
        lw = 0.7 + 2.6 * min(1.0, abs(score) / max_abs)
        rad = 0.05 * math.sin(hash((p, f)) % 5)
        ax.annotate(
            "",
            xy=pos[f],
            xytext=pos[p],
            arrowprops=dict(
                arrowstyle="-",
                color=color,
                lw=lw,
                alpha=0.58,
                connectionstyle=f"arc3,rad={rad:.2f}",
            ),
        )

    def draw_nodes(prefix: str, items: list[str], face: str, edge: str, size: int) -> None:
        for item in items:
            key = f"{prefix}:{item}"
            if key not in pos:
                continue
            x, y = pos[key]
            ax.scatter([x], [y], s=size, facecolor=face, edgecolor=edge, linewidth=1.4, zorder=5)
            label = _short_label(item, 30 if prefix != "func" else 34)
            ha = "right" if prefix == "cell" else "center" if prefix == "pert" else "left"
            dx = -0.025 if prefix == "cell" else 0 if prefix == "pert" else 0.025
            ax.text(x + dx, y, label, ha=ha, va="center", fontsize=9, zorder=6)

    draw_nodes("cell", cells, "#dbeafe", "#2563eb", 360)
    draw_nodes("pert", perts, "#fef3c7", "#d97706", 420)
    draw_nodes("func", funcs, "#f3f4f6", "#4b5563", 260)

    ax.plot([], [], color="#c94747", lw=3, label="activated edge")
    ax.plot([], [], color="#2f6fa3", lw=3, label="suppressed edge")
    ax.plot([], [], color="#9aa3ad", lw=1.4, label="cell-to-pert route")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.06), ncol=3, frameon=False, fontsize=9)
    ax.set_title(case_id, fontsize=14, weight="bold", pad=18)

    output_dir.mkdir(parents=True, exist_ok=True)
    out_png = output_dir / f"{case_id}_three_layer_route_graph.png"
    out_pdf = output_dir / f"{case_id}_three_layer_route_graph.pdf"
    fig.savefig(out_png, dpi=220, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)
    return out_png


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--case-dir", type=Path, required=True)
    parser.add_argument("--case-id", required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-functions", type=int, default=10)
    parser.add_argument("--max-edges-per-route", type=int, default=3)
    args = parser.parse_args()
    out = render_case(
        args.case_dir,
        args.case_id,
        args.output_dir,
        max_functions=args.max_functions,
        max_edges_per_route=args.max_edges_per_route,
    )
    print(out)


if __name__ == "__main__":
    main()


In [ ]:
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from pandas.errors import EmptyDataError

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "DejaVu Sans"


def clean_label(text: str) -> str:
    return str(text).replace("HALLMARK_", "").replace("_", " ").replace("  ", " ").strip()


def short_label(text: str, max_len: int = 24) -> str:
    text = clean_label(text)
    return text if len(text) <= max_len else text[: max_len - 1] + "..."


def wrapped_label(text: str, max_line: int = 16, max_lines: int = 2) -> str:
    text = str(text).replace("HALLMARK_", "").replace("_", " ").replace("  ", " ").strip()
    words = text.split()
    lines: list[str] = []
    current = ""
    for word in words:
        candidate = word if not current else f"{current} {word}"
        if len(candidate) <= max_line:
            current = candidate
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1][: max_line - 1].rstrip() + "..."
    return "\n".join(lines)


def layer_positions(items: list[str], y: float, x0: float = 0.08, x1: float = 0.92) -> dict[str, tuple[float, float]]:
    if not items:
        return {}
    if len(items) == 1:
        return {items[0]: ((x0 + x1) / 2, y)}
    step = (x1 - x0) / (len(items) - 1)
    return {item: (x0 + i * step, y) for i, item in enumerate(items)}


def consensus_top3_terms(ranked: pd.DataFrame) -> list[str]:
    activated = ranked[ranked["score"] > 0].sort_values("score", ascending=False).head(3)
    suppressed = ranked[ranked["score"] < 0].sort_values("score", ascending=True).head(3)
    return list(activated["label"].astype(str)) + list(suppressed["label"].astype(str))


def render_case(case_dir: Path, case_id: str, output_dir: Path) -> Path:
    route_path = case_dir / f"{case_id}_route_summary.csv"
    rf_path = case_dir / f"{case_id}_route_function_results.csv"
    selected_scores_path = case_dir / f"{case_id}_selected_function_route_scores.csv"
    ranked_path = case_dir / f"{case_id}_ranked_results.csv"
    routes = pd.read_csv(route_path)
    ranked = pd.read_csv(ranked_path)

    routes = routes[routes["status"].eq("executed")].copy()
    if routes.empty:
        routes = pd.read_csv(route_path)

    funcs = consensus_top3_terms(ranked)
    if selected_scores_path.exists() and selected_scores_path.stat().st_size > 0:
        try:
            rf = pd.read_csv(selected_scores_path)
        except EmptyDataError:
            rf = pd.read_csv(rf_path)
    else:
        rf = pd.read_csv(rf_path)
    rf = rf[rf["label"].astype(str).isin(funcs)].copy()

    cells = list(dict.fromkeys(routes["cell"].dropna().astype(str)))
    perts = list(dict.fromkeys(routes["perturbation"].dropna().astype(str)))

    pos: dict[str, tuple[float, float]] = {}
    pos.update({f"cell:{k}": v for k, v in layer_positions(cells, 0.78, x0=0.16, x1=0.92).items()})
    pos.update({f"pert:{k}": v for k, v in layer_positions(perts, 0.49, x0=0.14, x1=0.94).items()})
    pos.update({f"func:{k}": v for k, v in layer_positions(funcs, 0.18, x0=0.12, x1=0.96).items()})

    width = max(8.2, 1.05 * max(len(cells), len(perts), len(funcs)))
    fig, ax = plt.subplots(figsize=(width, 9.2))
    ax.set_axis_off()
    ax.set_xlim(-0.10, 1)
    ax.set_ylim(0, 1)

    ax.text(-0.085, 0.78, "Cell\ncontext", ha="left", va="center", fontsize=10, weight="bold", color="#374151")
    ax.text(-0.085, 0.49, "Perturbation\nevidence", ha="left", va="center", fontsize=10, weight="bold", color="#374151")
    ax.text(-0.085, 0.18, "Top consensus\nfunctional\nprograms", ha="left", va="center", fontsize=10, weight="bold", color="#374151")

    # Cell -> perturbation route edges. Semantic/proxy routes use dashed lines.
    for _, row in routes.iterrows():
        c = f"cell:{row['cell']}"
        p = f"pert:{row['perturbation']}"
        if c not in pos or p not in pos:
            continue
        match_type = str(row.get("perturbation_match_type", ""))
        linestyle = "-" if match_type.startswith(("user", "normalized")) else "--"
        ax.annotate(
            "",
            xy=pos[p],
            xytext=pos[c],
            arrowprops=dict(arrowstyle="-", color="#9ca3af", lw=1.4, linestyle=linestyle, alpha=0.62),
        )

    max_abs = max(1.0, float(rf["score"].abs().max() if not rf.empty else 1.0))
    # Collapse route/cell-level scores to one perturbation-function edge.
    if not rf.empty:
        rf = rf.groupby(["perturbation", "label"], as_index=False)["score"].mean()
        rf = rf[rf["score"].abs() > 1e-12].copy()
    for _, row in rf.iterrows():
        p = f"pert:{row['perturbation']}"
        f = f"func:{row['label']}"
        if p not in pos or f not in pos:
            continue
        score = float(row["score"])
        color = "#c94747" if score > 0 else "#2f6fa3"
        lw = 0.7 + 2.8 * min(1.0, abs(score) / max_abs)
        ax.annotate(
            "",
            xy=pos[f],
            xytext=pos[p],
            arrowprops=dict(arrowstyle="-", color=color, lw=lw, alpha=0.5),
        )

    def draw_node(
        x: float,
        y: float,
        label: str,
        face: str,
        edge: str,
        size: int,
        fs: int = 10,
        max_line: int = 16,
        yoff: float = 0.055,
    ) -> None:
        ax.scatter([x], [y], s=size, facecolor=face, edgecolor=edge, linewidth=1.7, zorder=4)
        ax.text(x, y - yoff, wrapped_label(label, max_line=max_line), ha="center", va="top", fontsize=fs, zorder=5, linespacing=1.05)

    for cell in cells:
        draw_node(*pos[f"cell:{cell}"], cell, "#dbeafe", "#2563eb", 390, fs=9, max_line=12)
    for pert in perts:
        draw_node(*pos[f"pert:{pert}"], pert, "#fef3c7", "#d97706", 420, fs=9, max_line=12)
    for func in funcs:
        score = float(ranked.loc[ranked["label"].astype(str).eq(func), "score"].iloc[0])
        # Neutral node, small ring tint only marks final consensus sign; route direction remains edge-coded.
        edge = "#c94747" if score > 0 else "#2f6fa3"
        draw_node(*pos[f"func:{func}"], func, "#f9fafb", edge, 330, fs=8, max_line=15, yoff=0.045)

    ax.plot([], [], color="#c94747", lw=3, label="activated evidence edge")
    ax.plot([], [], color="#2f6fa3", lw=3, label="suppressed evidence edge")
    ax.plot([], [], color="#9ca3af", lw=1.4, label="cell-to-pert route")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.04), ncol=3, frameon=False, fontsize=10)
    ax.set_title(case_id, fontsize=15, weight="bold", pad=16)

    output_dir.mkdir(parents=True, exist_ok=True)
    out_png = output_dir / f"{case_id}_vertical_top3_route_graph.png"
    out_pdf = output_dir / f"{case_id}_vertical_top3_route_graph.pdf"
    fig.savefig(out_png, dpi=240, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)
    return out_png


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--case-dir", type=Path, required=True)
    parser.add_argument("--case-id", required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    args = parser.parse_args()
    print(render_case(args.case_dir, args.case_id, args.output_dir))


if __name__ == "__main__":
    main()
